In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn import model_selection
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [2]:
data_file = Path('..') / 'model' / 'SUPERSTORE_CLUSTERING.csv'
df = pd.read_csv(data_file, sep=',')

In [3]:
df.head()

,CUSTOMER_ID,ORDER_DATE,SHIP_MODE,CATEGORY,REGION,SEGMENT,NET_SALES,PROFIT,NET_SALES_SCALED,PROFIT_SCALED,CLUSTER
0,AA-10315,2019-03-31,STANDARD CLASS,OFFICE SUPPLIES,WEST,CONSUMER,204.7000,28.8658,-0.802781,-0.291871,0
1,AA-10375,2019-04-21,STANDARD CLASS,OFFICE SUPPLIES,WEST,CONSUMER,1171.1996,91.3899,-1.674506,-0.575691,0
2,AA-10480,2019-05-04,SAME DAY,FURNITURE,EAST,CONSUMER,357.4148,63.0594,-0.759041,-0.142015,0
3,AA-10645,2019-12-01,FIRST CLASS,FURNITURE,EAST,CONSUMER,291.2400,38.4204,-0.777994,-0.249997,0
4,AB-10015,2019-02-18,STANDARD CLASS,OFFICE SUPPLIES,CENTRAL,CONSUMER,59.1680,2.9553,-0.557327,-0.265966,0


In [4]:
df_predict = df[['ORDER_DATE', 'SHIP_MODE', 'PROFIT_SCALED', 'NET_SALES_SCALED', 'REGION', 'CLUSTER']].copy()

In [5]:
df_predict = pd.get_dummies(
    df_predict,
    columns=['REGION'])

In [6]:
df_predict['SHIP_MODE'].value_counts()

SHIP_MODE
STANDARD CLASS    445
SECOND CLASS      162
FIRST CLASS       119
SAME DAY           44
Name: count, dtype: int64

In [7]:
state_map = {
        'STANDARD CLASS': '0',
        'SECOND CLASS': '1',
        'FIRST CLASS': '2',
        'SAME DAY': '3'
}

df_predict['SHIP_MODE'] = df_predict['SHIP_MODE'].map(state_map)
df_predict['SHIP_MODE'] = df_predict['SHIP_MODE'].astype('Int64')

In [8]:
df_predict['ORDER_DATE'] = pd.to_datetime(df_predict['ORDER_DATE'], errors='coerce')
df_predict['ORDER_YEAR'] = df_predict['ORDER_DATE'].dt.year

df_predict = df_predict.drop(columns=['ORDER_DATE'])

In [9]:
df_predict.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770 entries, 0 to 769
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SHIP_MODE         770 non-null    Int64  
 1   PROFIT_SCALED     770 non-null    float64
 2   NET_SALES_SCALED  770 non-null    float64
 3   CLUSTER           770 non-null    int64  
 4   REGION_CENTRAL    770 non-null    bool   
 5   REGION_EAST       770 non-null    bool   
 6   REGION_SOUTH      770 non-null    bool   
 7   REGION_WEST       770 non-null    bool   
 8   ORDER_YEAR        770 non-null    int32  
dtypes: Int64(1), bool(4), float64(2), int32(1), int64(1)
memory usage: 31.0 KB


In [10]:
df_predict.head()

,SHIP_MODE,PROFIT_SCALED,NET_SALES_SCALED,CLUSTER,REGION_CENTRAL,REGION_EAST,REGION_SOUTH,REGION_WEST,ORDER_YEAR
0,0,-0.291871,-0.802781,0,False,False,False,True,2019
1,0,-0.575691,-1.674506,0,False,False,False,True,2019
2,3,-0.142015,-0.759041,0,False,True,False,False,2019
3,2,-0.249997,-0.777994,0,False,True,False,False,2019
4,0,-0.265966,-0.557327,0,True,False,False,False,2019


In [11]:
dfs_clusters = {
    cluster: df_predict[df_predict['CLUSTER'] == cluster]
    for cluster in df_predict['CLUSTER'].unique()
}

In [16]:
dfs_clusters

{np.int64(0):      SHIP_MODE  PROFIT_SCALED  NET_SALES_SCALED  CLUSTER  REGION_CENTRAL  \
 0            0      -0.291871         -0.802781        0           False   
 1            0      -0.575691         -1.674506        0           False   
 2            3      -0.142015         -0.759041        0           False   
 3            2      -0.249997         -0.777994        0           False   
 4            0      -0.265966         -0.557327        0            True   
 ..         ...            ...               ...      ...             ...   
 761          0      -0.113374         -0.254406        0           False   
 764          0      -0.485786         -1.569046        0           False   
 765          0      -0.743573         -2.153460        0            True   
 768          1      -0.190285         -0.527782        0           False   
 769          0      -0.191561         -0.502326        0           False   
 
      REGION_EAST  REGION_SOUTH  REGION_WEST  ORDER_YEAR  
 0

In [12]:
X_train = []
X_test = []
y_train = []
y_test = []

for i in range(0, 3):

    X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
        dfs_clusters[i].drop('PROFIT_SCALED', axis=1),
        dfs_clusters[i]['PROFIT_SCALED'],
        test_size=0.2,
        random_state=0
    )

    X_train.append(X_train_i)
    X_test.append(X_test_i)
    y_train.append(y_train_i)
    y_test.append(y_test_i)
    

In [13]:
models = {
    'XGBOOST': XGBRegressor(random_state=0),
    'REGRESSAO LINEAR': LinearRegression(),
    'RANDOM FOREST': RandomForestRegressor(random_state=0),
    'SVR': SVR(),
    'LIGHTGBM': lgb.LGBMRegressor(
        random_state=0,
        force_col_wise=True,
        verbose=-1,
        n_jobs=1
    )
}

for d in [2, 3, 4]:
    models[f'REGRESSAO POLINOMIAL GRAU {d}'] = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('model', LinearRegression())
    ])

kfold = model_selection.KFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

print('=' * 40)
print('RESULTADOS DA EXPLORACAO (REGRESSAO)')
print('=' * 40, '\n')

results = []

for i in range(0, 3):
    results = []
    for name, model in models.items():
        scores = model_selection.cross_validate(
            model,
            X_train[i],
            y_train[i],
            scoring=['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
            cv=kfold,
            n_jobs=1
        )

        mse = round(-scores['test_neg_mean_squared_error'].mean(), 4)
        mae = round(-scores['test_neg_mean_absolute_error'].mean(), 4)
        r2 = round(scores['test_r2'].mean(), 4)

        result = {
            'NAME': name,
            'MSE': mse,
            'MAE': mae,
            'R2': r2
        }
        results.append(result)

    print('CLUSTER ', i)
    results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
    display(results_df)



RESULTADOS DA EXPLORACAO (REGRESSAO)

CLUSTER  0


,NAME,MSE,MAE,R2
0,REGRESSAO LINEAR,0.0079,0.0701,0.6764
1,REGRESSAO POLINOMIAL GRAU 2,0.0086,0.0721,0.6494
2,LIGHTGBM,0.0094,0.0742,0.6138
3,RANDOM FOREST,0.0097,0.0763,0.6007
4,REGRESSAO POLINOMIAL GRAU 3,0.0106,0.0759,0.5852
5,REGRESSAO POLINOMIAL GRAU 4,0.0115,0.0759,0.5566
6,XGBOOST,0.0122,0.0843,0.4906
7,SVR,0.0276,0.1237,-0.0344


CLUSTER  1


,NAME,MSE,MAE,R2
0,REGRESSAO LINEAR,0.0102,0.0773,0.6683
1,RANDOM FOREST,0.0107,0.0762,0.6520
2,REGRESSAO POLINOMIAL GRAU 2,0.0111,0.0813,0.6378
3,LIGHTGBM,0.0114,0.0801,0.6305
4,XGBOOST,0.0146,0.0901,0.5406
5,REGRESSAO POLINOMIAL GRAU 3,0.0140,0.0901,0.5314
6,SVR,0.0342,0.1404,-0.0594
7,REGRESSAO POLINOMIAL GRAU 4,0.0345,0.1140,-0.2236


CLUSTER  2


,NAME,MSE,MAE,R2
0,REGRESSAO LINEAR,0.0081,0.0712,0.6357
1,RANDOM FOREST,0.0097,0.0750,0.6066
2,REGRESSAO POLINOMIAL GRAU 2,0.0095,0.0791,0.5583
3,LIGHTGBM,0.0097,0.0764,0.5465
4,XGBOOST,0.0128,0.0851,0.4930
5,SVR,0.0285,0.1327,-0.0878
6,REGRESSAO POLINOMIAL GRAU 3,2.3546,0.3192,-61.0906
7,REGRESSAO POLINOMIAL GRAU 4,3.6641,0.5088,-102.4905


Temos em todos os modelos que REGRESSÃO LINEAR é o modelo preditivo mais promissor.

In [14]:
regularization_models = {
    'RIDGE': Ridge(max_iter=5000),
    'LASSO': Lasso(max_iter=5000),
    'ELASTICNET': ElasticNet(max_iter=5000, tol=1e-3)
}

param_grids = {
    'RIDGE': {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    'LASSO': {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    'ELASTICNET': {
        'alpha': [0.001, 0.01, 0.1, 1, 10],
        'l1_ratio': [0.35, 0.45, 0.55, 0.65]
    }
}

results_tuning = []

for i in range(0, 3):
    for model_name, model in regularization_models.items():
        param_grid = param_grids[model_name]

        grid_search = model_selection.GridSearchCV(
            model,
            param_grid,
            scoring='r2',
            cv=kfold,
            n_jobs=-1,
            verbose=0
        )

        grid_search.fit(X_train[i], y_train[i])
        best_estimator = grid_search.best_estimator_

        r2_cruzada = round(grid_search.best_score_, 4)
        r2_teste = round(best_estimator.score(X_test[i], y_test[i]), 4)

        result = {
            'CLUSTER': i,
            'MODEL': model_name,
            'BEST_PARAMS': grid_search.best_params_,
            'CV_R2': r2_cruzada,
            'TEST_R2': r2_teste,
            'R2_GAP': round(abs(r2_teste - r2_cruzada), 4),
            'ESTIMATOR': best_estimator
        }
        results_tuning.append(result)

results_tuning_df = pd.DataFrame(results_tuning)
results_tuning_view = results_tuning_df.drop(columns=['ESTIMATOR']).sort_values(
    ['CLUSTER', 'TEST_R2'], ascending=[True, False]
).reset_index(drop=True)

display(results_tuning_view)



,CLUSTER,MODEL,BEST_PARAMS,CV_R2,TEST_R2,R2_GAP
0,0,LASSO,{'alpha': 0.001},0.6785,0.6947,0.0162
1,0,RIDGE,{'alpha': 1},0.6770,0.6943,0.0173
2,0,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.6788,0.6829,0.0041
3,1,RIDGE,{'alpha': 1},0.6712,0.6726,0.0014
4,1,LASSO,{'alpha': 0.001},0.6744,0.6697,0.0047
5,1,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.6789,0.6696,0.0093
6,2,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.6484,0.5250,0.1234
7,2,LASSO,{'alpha': 0.001},0.6462,0.4889,0.1573
8,2,RIDGE,{'alpha': 1},0.6428,0.4737,0.1691


In [15]:
best_models_df = (
    results_tuning_df
    .sort_values(['CLUSTER', 'TEST_R2'], ascending=[True, False])
    .groupby('CLUSTER', as_index=False)
    .first()
    .reset_index(drop=True)
    .rename(
        columns={
            'MODEL': 'BEST_MODEL',
            'BEST_PARAMS': 'BEST_PARAMS',
            'CV_R2': 'BEST_CV_R2',
            'TEST_R2': 'BEST_TEST_R2',
            'R2_GAP': 'BEST_R2_GAP'
        }
    )
)

best_models_view = best_models_df[[
    'CLUSTER', 'BEST_MODEL', 'BEST_PARAMS', 'BEST_CV_R2', 'BEST_TEST_R2', 'BEST_R2_GAP'
]]

display(best_models_view)

,CLUSTER,BEST_MODEL,BEST_PARAMS,BEST_CV_R2,BEST_TEST_R2,BEST_R2_GAP
0,0,LASSO,{'alpha': 0.001},0.6785,0.6947,0.0162
1,1,RIDGE,{'alpha': 1},0.6712,0.6726,0.0014
2,2,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.6484,0.5250,0.1234
